### <span style=color:blue> Loading Calendar data from csv into local MongoDB    </span>

In [1]:
import sys
import json
import csv
import yaml

import importlib

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime
# with the above choices, the imported datetime.time(2023,07,01) is recognized
# from datetime import date
# from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

# Create an utilities file util.py in a folder benchmarking and import it
sys.path.append('ECS116-HELPER-FUNCTIONS')
# import util as util
import util

<span style=color:blue>Getting mongodb connection set up</span>

In [2]:
from pymongo import MongoClient

client = MongoClient()
# could have written client = MongoClient("localhost", 27017)
#                 or client = MongoClient("mongodb://localhost:27017/")

<span style=color:blue>Getting access to airbnb database, and setting up collection "cal" to hold the calendar data in mongodb</span>

In [3]:
# I have (or will have) a database "airbnb"
db = client.airbnb


print('The list of all databases currently in the MongoDB client is:')
print(client.list_database_names())

print('\nThe list of all collections in the airbnb database is:')
print(db.list_collection_names())
# Note: calendar may not show up yet; it is created only when a first document is inserted into it

The list of all databases currently in the MongoDB client is:
['admin', 'airbnb', 'config', 'local', 'test']

The list of all collections in the airbnb database is:
['listings_small', 'listings_test', 'listings', 'listings_with_calendar', 'listings_with_reviews_duplicate', 'calendar']


<span style=color:blue>Loading contents of calendar csv file into a dataframe</span>

<span style=color:blue>The system will give a warning, but it appears safe to ignore it.</span>

<span style=color:blue>This may take 30 or 60 seconds to run; wait for the printed message output</span>

In [4]:
filename = '/Users/rick/DM-for-DS-2025/DATA-SETS/AirBnB/New-York-City/calendar.csv'

# Using partial list of dtypes, so that first several fields are interpreted as strings
# As for the date and available fields (intended as date type and boolean, respectively,
#    we import as strings and convert in the data frame
dtype = {"listing_id": str, "date": str, "available": str, 
        "price": str, "adjusted_price": str}
# note including these, because the null values make trouble:  , "minimum_nights": int, "maximum_nights": int}

# the csv has nulls in "adjusted_price", which has type str,. so including keep_default_na=False, 
#    see https://stackoverflow.com/questions/10867028/get-pandas-read-csv-to-read-empty-values-as-empty-string-instead-of-nan
         
df = pd.read_csv(filename, dtype=dtype, keep_default_na=False)

print('\nThis command has apparently succeeded')

/var/folders/3l/gd1qj_mw8xl3gm001s6w9h180000gp/T/ipykernel_23135/2457680385.py:13: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename, dtype=dtype, keep_default_na=False)



This command has apparently succeeded


In [5]:
print('The datatypes for the fields of df are:')
print(df.dtypes)

print('\nThe first few rows of df are:')
print(df.head())

The datatypes for the fields of df are:
listing_id        object
date              object
available         object
price             object
adjusted_price    object
minimum_nights    object
maximum_nights    object
dtype: object

The first few rows of df are:
  listing_id        date available    price adjusted_price minimum_nights  \
0       2595  2025-03-03         t  $225.00                            30   
1       2595  2025-03-04         t  $225.00                            30   
2       2595  2025-03-05         t  $225.00                            30   
3       2595  2025-03-06         t  $225.00                            30   
4       2595  2025-03-07         t  $225.00                            30   

  maximum_nights  
0           1125  
1           1125  
2           1125  
3           1125  
4           1125  


<span style=color:blue>Function to convert date strings into datetimes. This is similar to, but different from, the function convert_date_to_datetime() in 1--Loading-Local-MongoDB-with-Listings-&-Reviews--v01.ipynb.  The current function takes a string as input, the other function takes a date object as input.</span>

<span style=color:blue>This function had been developed when working with the listings join reviews data, where there were some NULL values.  There are no NULL values in the calendar data.</span>

<span style=color:blue>A better sw engineering practice might be to include this function into the util.py file, but leaving it here to make the notebook more self-contained.</span>

In [6]:
# also converts NaT to None, because MongoDB does not recognize NaT
def convert_date_str_to_datetime(dt):
    if pd.isnull(dt):           # tests whether dt is None, NaN, or DaT (not a date)
        return None
    else:
        year = int(dt[0:4])
        month = int(dt[5:7])
        day = int(dt[8:10])
        # print(year, month, day)
        temp = datetime(year, month, day)
        ts = temp.timestamp()
        new_dt = datetime.fromtimestamp(ts)
        return new_dt

new_dt = convert_date_str_to_datetime('2024-05-23')
print(type(new_dt))
print(new_dt)


<class 'datetime.datetime'>
2024-05-23 00:00:00


<span style=color:blue>Function to convert the values of field "available" to booleans </span>

In [7]:
def convert_tf_to_boolean(val):
    if val == 't':
        return True
    elif val == 'f':
        return False
    else:
        return None

print(convert_tf_to_boolean('t'), convert_tf_to_boolean('f'), convert_tf_to_boolean('foo'))


True False None


<span style=color:blue>Cleaning up the values in df, to be more compatible with MongoDB. (This may take a minute or 2; wait for the printed output.) </span>

In [8]:
df['date'] = df['date'].apply(convert_date_str_to_datetime)
df['available'] = df['available'].apply(convert_tf_to_boolean)

print(df.head())

  listing_id       date  available    price adjusted_price minimum_nights  \
0       2595 2025-03-03       True  $225.00                            30   
1       2595 2025-03-04       True  $225.00                            30   
2       2595 2025-03-05       True  $225.00                            30   
3       2595 2025-03-06       True  $225.00                            30   
4       2595 2025-03-07       True  $225.00                            30   

  maximum_nights  
0           1125  
1           1125  
2           1125  
3           1125  
4           1125  


In [9]:
print('The datatypes for columns of df are:\n')

print(df.dtypes)

The datatypes for columns of df are:

listing_id                object
date              datetime64[ns]
available                   bool
price                     object
adjusted_price            object
minimum_nights            object
maximum_nights            object
dtype: object


<span style=color:blue>Cleaning and changing data type of price column.  (Not bothering with adjusted_price column, which appears to be uniformly NULL.   </span>

In [10]:
# need to strip the leading '$' from the price value, and remove commas
df['price'] = df['price'].apply(lambda x:x.replace('$','').replace(',',''))

# converting price which is string to numeric
df['price'] = pd.to_numeric(df['price']) 

print(type(df.loc[0,'price']))
print(df.head())

<class 'numpy.float64'>
  listing_id       date  available  price adjusted_price minimum_nights  \
0       2595 2025-03-03       True  225.0                            30   
1       2595 2025-03-04       True  225.0                            30   
2       2595 2025-03-05       True  225.0                            30   
3       2595 2025-03-06       True  225.0                            30   
4       2595 2025-03-07       True  225.0                            30   

  maximum_nights  
0           1125  
1           1125  
2           1125  
3           1125  
4           1125  


### <span style=color:blue>Now working to fill listings_with_cal with the dataframe df.  After that we will use an agg function to produce a collection that contains listings and for each listing an array of dates that it is available,   </span>

<span style=color:blue>In this notebook we work directly with the full calendar data set.  You might want to practice with a subset of the data, e.g., by using df_small = df.iloc[0:5000].</span>

<span style=color:blue>First step is to load the df into a dict. Takes a minute or 2.</span>

In [11]:
time1 = datetime.now()
dict_full = df.to_dict('records')
time2 = datetime.now()
print(f'Time to perform this operation was {util.time_diff(time1,time2)} seconds.')

Time to perform this operation was 146.703789 seconds.


<span style=color:blue>Now loading the dictionary into MongoDB. Takes about 2 to 4 minutes.</span>

In [12]:
# The following empties out the calendar collection; useful if making a fresh start
db.calendar.drop()

print(f"Size of dict_full is: {len(dict_full)}")

time1 = datetime.now()
result = db.calendar.insert_many(dict_full)
time2 = datetime.now()
print(f'\nTime to perform this operation was {util.time_diff(time1,time2)} seconds.')
print(f'\nTime to perform this operation was {util.time_diff(time1,time2)/60} minutes.')

# between about 2 and 4 minutes

print(f'\nNumber of docs in db.calendar is {db.calendar.count_documents({})}')

print('\nHere are the last 5 objects in the calendar collection:')
outdocs = []
for o in result.inserted_ids[-5:]:
    outdocs.append(db.calendar.find_one({ '_id': o}))
pprint.pp(outdocs)

Size of dict_full is: 13661894

Time to perform this operation was 419.628662 seconds.

Time to perform this operation was 6.993811033333333 minutes.

Number of docs in db.calendar is 13661894

Here are the last 5 objects in the calendar collection:
[{'_id': ObjectId('682c0e4a9381acc64b066425'),
  'listing_id': '1366723228243064949',
  'date': datetime.datetime(2026, 2, 25, 0, 0),
  'available': True,
  'price': 73.0,
  'adjusted_price': '',
  'minimum_nights': 30,
  'maximum_nights': 365},
 {'_id': ObjectId('682c0e4a9381acc64b066426'),
  'listing_id': '1366723228243064949',
  'date': datetime.datetime(2026, 2, 26, 0, 0),
  'available': True,
  'price': 73.0,
  'adjusted_price': '',
  'minimum_nights': 30,
  'maximum_nights': 365},
 {'_id': ObjectId('682c0e4a9381acc64b066427'),
  'listing_id': '1366723228243064949',
  'date': datetime.datetime(2026, 2, 27, 0, 0),
  'available': True,
  'price': 73.0,
  'adjusted_price': '',
  'minimum_nights': 30,
  'maximum_nights': 365},
 {'_id': Obj

In [13]:
# Sanity check that we didn't lose any records
print(len(dict_full))
print(db.calendar.count_documents({}))

13661894
13661894


<span style=color:blue>When you run your pipeline on the full calendar dataset, it may take about 3 minutes or more     </span>

In [14]:
# making sure that listings_with_calendar is empty
db.listings_with_calendar.drop()

"""
pipeline = [

############################################################    
#                                                          #
#    you need to fill this in as part of the assignment    #
#                                                          #
############################################################    
    
]
"""

time1 = datetime.now()
test1 = db.calendar.aggregate(pipeline)
time2 = datetime.now()
diff = util.time_diff(time1, time2)

print(f"The type of 'test1' is {type(test1)}, but it's empty.")

print('\nTime it took was:', format(diff, '.4f'), 'seconds.')

# checking that collection listings_with_calendar was created
print('\nThe full set of collection names is now:')
print(db.list_collection_names())

The type of 'test1' is <class 'pymongo.synchronous.command_cursor.CommandCursor'>, but it's empty.

Time it took was: 190.9551 seconds.

The full set of collection names is now:
['listings_small', 'listings_test', 'listings_with_calendar', 'listings', 'calendar', 'listings_with_reviews_duplicate']


<span style=color:blue>Finding the count of listing documents in listings_with_calendar.  Why is this different from 37,434, which is the total number of listings?     </span>

In [15]:
count = db.listings_with_calendar.count_documents({})
print(count)

37431


In [16]:
pprint.pp(db.listings_with_calendar.find_one())

{'_id': '10000070',
 'average_price': 85.0,
 'first_available_date': datetime.datetime(2025, 3, 3, 0, 0),
 'last_available_date': datetime.datetime(2026, 3, 2, 0, 0),
 'dates_list': [{'date': datetime.datetime(2025, 3, 3, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 4, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 5, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 6, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30

<span style=color:blue>Here is one way to fetch the data types of fields of documents in a collection.  (This works for db.listings_with_calendar because all documents in the collection have the same structure.)    </span>

In [17]:
# Retrieve a single document from the collection
doc = db.listings_with_calendar.find_one()

# Iterate over the keys of the retrieved document and retrieve the datatype of each value
for key in doc.keys():
    print(key, type(doc[key]))
for key in doc['dates_list'][0]:
    print(key, type(doc['dates_list'][0][key]))

_id <class 'str'>
average_price <class 'float'>
first_available_date <class 'datetime.datetime'>
last_available_date <class 'datetime.datetime'>
dates_list <class 'list'>
date <class 'datetime.datetime'>
available <class 'bool'>
price <class 'float'>
minimum_nights <class 'int'>
maximum_nights <class 'int'>


<span style=color:blue>As you may recall from the notebook "Loading-Local-MongoDB-with-Listings-&-Reviews--vXX.ipynb", in general, you cannot fetch documents from MongoDB and write them into json files on your machine.  In the next 2 cells we work to create a function that transforms documents in db.listings_with_calendar into dicts that can be written out to json files.   </span>

In [18]:
doc = db.listings_with_calendar.find_one()
pprint.pp(doc)

{'_id': '10000070',
 'average_price': 85.0,
 'first_available_date': datetime.datetime(2025, 3, 3, 0, 0),
 'last_available_date': datetime.datetime(2026, 3, 2, 0, 0),
 'dates_list': [{'date': datetime.datetime(2025, 3, 3, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 4, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 5, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 6, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30

<span style=color:blue>The basic idea is to convert data objects that json cannot handle. In this example, the troublesome data type is datetime, which is replaced with a string with a standard format.  Some of the dates are within the dates_list array, and so we have to navigate inside those arrays.     </span>

In [19]:
def convert_lwc_to_json(doc):
    doc_new = {}
    for key in ['_id', 'average_price']:
        doc_new[key] = doc[key]
    for key in ['first_available_date', 'last_available_date']:
        doc_new[key] = doc[key].strftime('%Y-%m-%d')
    dlist = []
    for d in doc['dates_list']:
        d_new = {}
        d_new['date'] = d['date'].strftime('%Y-%m-%d')
        for key in ['price', 'minimum_nights', 'maximum_nights', 'available']:
            d_new[key] = d[key]
        dlist.append(d_new)
    doc_new['dates_list'] = dlist
    return doc_new

pprint.pp(convert_lwc_to_json(doc))

{'_id': '10000070',
 'average_price': 85.0,
 'first_available_date': '2025-03-03',
 'last_available_date': '2026-03-02',
 'dates_list': [{'date': '2025-03-03',
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30,
                 'available': True},
                {'date': '2025-03-04',
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30,
                 'available': True},
                {'date': '2025-03-05',
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30,
                 'available': True},
                {'date': '2025-03-06',
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30,
                 'available': True},
                {'date': '2025-03-07',
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum

<span style=color:blue>Will now fetch a small subset of db.listings_with_calendar, and write those documents into the json file "listings_with_calendar_subset_example.json".  You have to create a file somewhat like this for PA3 Step 2.    </span>

In [20]:
# first, creating an index on average_price, to speed up the next cell

db.listings_with_calendar.create_index('average_price')

'average_price_1'

In [27]:
# print(db.listings_with_calendar.count_documents({}))

# cursor = db.listings_with_calendar.find({'_id' : {'$regex' : '^11110.*$'}})

# dt = datetime.strptime('2026-03-02','%Y-%m-%d')

cursor = db.listings_with_calendar.find({
    'average_price' : {
        '$gte' : 18370
    }
})
    
l = list(cursor)
print(len(l))

157


<span style=color:blue>Fetching the contents of cursor, and converting to json   </span>

In [28]:
# need to recompute the cursor, because testing the length in 
#    previous cell ran through list the cursor was pointing at
cursor = db.listings_with_calendar.find({
    'average_price' : {
        '$gte' : 18370
    }
})

output = []

for doc in cursor:
    output.append(convert_lwc_to_json(doc))

print(len(output))

157


In [30]:
for doc in output[0:3]:
    pprint.pp(doc)

{'_id': '1142074768335738644',
 'average_price': 18370.0,
 'first_available_date': '2025-03-03',
 'last_available_date': '2026-03-02',
 'dates_list': [{'date': '2025-03-03',
                 'price': 18370.0,
                 'minimum_nights': 30,
                 'maximum_nights': 365,
                 'available': False},
                {'date': '2025-03-04',
                 'price': 18370.0,
                 'minimum_nights': 30,
                 'maximum_nights': 365,
                 'available': True},
                {'date': '2025-03-05',
                 'price': 18370.0,
                 'minimum_nights': 30,
                 'maximum_nights': 365,
                 'available': True},
                {'date': '2025-03-06',
                 'price': 18370.0,
                 'minimum_nights': 30,
                 'maximum_nights': 365,
                 'available': True},
                {'date': '2025-03-07',
                 'price': 18370.0,
                 'minimum_nigh

In [31]:
# Writing dict to a json file into a json file in a subdirectory
# Also putting this function into my util.py

def write_dict_to_dir_json(dict, dir, filename):
    with open(dir + filename, 'w') as fp:
        json.dump(dict, fp)

dir = '/Users/rick/DM-for-DS-2025/PA3-OUTPUT/'
filename = 'listings_with_calendar_subset_avg_price_18370.json'
write_dict_to_dir_json(output, dir, filename)

